capture the location data from text column

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.affect_temp (
  id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  text STRING,
  matched STRING
);
INSERT INTO TABLE beneco_pipeline.gold.affect_temp(text, matched)
WITH temp AS (
  SELECT 
    concat('(?i)\\b(',concat_ws('|', collect_list(barangay)),')\\b') AS brgy
  FROM beneco_pipeline.silver.location_silver
)
SELECT 
  text,
  CASE 
    WHEN text RLIKE brgy THEN (
      array_join(regexp_extract_all(text, brgy, 0),', ')
    )
    WHEN text RLIKE 'Ambiong' THEN (
      CASE 
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Ambiong',1) RLIKE '(?i)La Trinidad' THEN 'Ambiong LT'
        ELSE 'Ambiong BC'
      END
    )
    WHEN text RLIKE 'Balili' THEN (
      CASE 
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Balili',1) RLIKE '(?i)La Trinidad' THEN 'Balili LT'
        ELSE 'Balili MK'
      END
    )
    WHEN text RLIKE 'Poblaci(o|ó)n' THEN (
      CASE 
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Atok' THEN 'Poblacion AT'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Bokod' THEN 'Poblacion BD'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Bakun' THEN 'Poblacion BN'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Buguias' THEN 'Poblacion BS'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Kapangan' THEN 'Poblacion Central'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Itogon' THEN 'Poblacion IT'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)La Trinidad' THEN 'Poblacion LT'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Mankayan' THEN 'Poblacion MK'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Sablan' THEN 'Poblacion SB'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Tuba' THEN 'Poblacion TB'
        ELSE 'Poblacion KB'
      END
    )    
    ELSE 'not matched'
  END AS matched_names
FROM beneco_pipeline.silver.interruption_silver AS ben
CROSS JOIN temp;

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.alias (
    alias STRING,
    loc_name STRING
);
INSERT INTO beneco_pipeline.gold.alias VALUES
('camp allen', 'Camp Henry I. Allen'),
('Saint Joseph','St. Joseph Village'),
('Saint Joseph Village','St. Joseph Village'),
('Bakakeng Norte','Bakakeng Norte/Sur'),
('Bakakeng Sur','Bakakeng Norte/Sur'),
('Bayan Park','Bayan Park Village'),
('Ambuclao','Ambuklao'),
('Sto. Nino- Slaughter Compound','Sto. Nino-Slaughter Compound'),
('Apugan-Loakan','Loakan Apugan'),
('San Luis', 'San Luis Village');

In [0]:
%sql
WITH ranked_matches AS (
  SELECT 
    target.text,
    source.loc_name,
    ROW_NUMBER() OVER (PARTITION BY target.text ORDER BY LENGTH(source.alias) DESC) AS rn
  FROM beneco_pipeline.gold.affect_temp AS target
  CROSS JOIN beneco_pipeline.gold.alias AS source
  WHERE LOWER(target.text) RLIKE concat('(?i)\\b(',LOWER(source.alias),')\\b') 
    AND target.matched = 'not matched'
)
MERGE INTO beneco_pipeline.gold.affect_temp AS target
USING (SELECT text, loc_name FROM ranked_matches WHERE rn = 1) AS source
ON target.text = source.text AND target.matched = 'not matched'
WHEN MATCHED THEN
  UPDATE SET target.matched = source.loc_name;

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.affect_gold(
  interruption_id BIGINT,
  location_id BIGINT,
  FOREIGN KEY (location_id) REFERENCES beneco_pipeline.gold.location_gold(id),
  FOREIGN KEY (interruption_id) REFERENCES beneco_pipeline.gold.interruption_gold(id)
);

INSERT INTO beneco_pipeline.gold.affect_gold
SELECT 
  i.id AS interruption_id,
  l.id AS location_id
FROM (
  SELECT 
    id,
    text,
    explode(split(trim(matched), ', ')) AS match
  FROM beneco_pipeline.gold.affect_temp
) AS at
JOIN beneco_pipeline.gold.interruption_gold AS i ON 
  i.text = at.text
JOIN beneco_pipeline.gold.location_gold AS l ON
  l.barangay = at.match;

DROP TABLE beneco_pipeline.gold.affect_temp;
DROP TABLE beneco_pipeline.gold.alias;
DROP TABLE beneco_pipeline.gold.matches;